# BERT WordPiece Fix — PyTorch Backend


## Cell 1 — Install Dependencies

In [1]:
!pip install -q "tokenizers==0.19.1" "transformers==4.44.2" "huggingface-hub==0.23.4"

In [2]:
import transformers
print(transformers.__file__)
print(transformers.__version__)

!pip show transformers
!find /usr -name "pytorch_utils.py" -path "*/transformers/*" 2>/dev/null

/usr/local/lib/python3.12/dist-packages/transformers/__init__.py
4.44.2
Name: transformers
Version: 4.44.2
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers
/usr/local/lib/python3.12/dist-packages/transformers/pytorch_utils.py


In [3]:
import sys, importlib

mods_to_remove = [k for k in sys.modules if 'transformers' in k]
for m in mods_to_remove:
    del sys.modules[m]

import subprocess
result = subprocess.run(['pip', 'show', 'transformers'], capture_output=True, text=True)
print(result.stdout)

import transformers.pytorch_utils as pu
print(dir(pu))
print("find_pruneable" in dir(pu))

Name: transformers
Version: 4.44.2
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers

['ALL_LAYERNORM_LAYERS', 'Callable', 'Conv1D', 'List', 'Optional', 'Set', 'Tuple', 'Union', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'apply_chunking_to_forward', 'find_pruneable_heads_and_indices', 'id_tensor_storage', 'inspect', 'is_torch_greater_or_equal_than_1_12', 'is_torch_greater_or_equal_than_1_13', 'is_torch_greater_or_equal_than_2_0', 

In [4]:
!pip show transformers | grep Version

Version: 4.44.2


## Cell 2 — Setup & Configuration

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import random
import os
import json
import warnings
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

DATASET_25MS_PATH = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'
DATASET_15MS_PATH = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'

RESULTS_25MS = '/content/drive/MyDrive/Detector_Results/baselines_25ms'
RESULTS_15MS = '/content/drive/MyDrive/Detector_Results/baselines_15ms'
os.makedirs(RESULTS_25MS, exist_ok=True)
os.makedirs(RESULTS_15MS, exist_ok=True)

SEEDS = [42, 123, 456, 789, 2024]
FEATURES = ['SNR', 'RSSI', 'PDR', 'Speed', 'Relative_Speed']
EPISODE_LENGTH = 60
WINDOW_SEC = 15

BERT_EPOCHS = 10
BERT_BATCH = 16
BERT_LR = 2e-5
MAX_LEN_TOK = 128

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Results dirs:')
print(f'  25 m/s: {RESULTS_25MS}')
print(f'  15 m/s: {RESULTS_15MS}')

Mounted at /content/drive
Device: cuda
Results dirs:
  25 m/s: /content/drive/MyDrive/Detector_Results/baselines_25ms
  15 m/s: /content/drive/MyDrive/Detector_Results/baselines_15ms


## Cell 3 — Pipeline Functions

In [6]:
def set_all_seeds(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

SPEED_RANGE = {15: (13, 17), 25: (23, 27)}

def load_dataset(path, target_speed):
    df = pd.read_csv(path)
    df['label'] = df['Scenario'].map({1: 0, 2: 1, 3: 2})
    df = df.sort_values(['Scenario', 'Time']).reset_index(drop=True)
    df['pseudo_run_id'] = (
        df['Speed'].round(2).astype(str) + '_' +
        df['Scenario'].astype(str) + '_' +
        (df['Time'] // EPISODE_LENGTH).astype(int).astype(str)
    )

    speed_low, speed_high = SPEED_RANGE[target_speed]
    df = df[(df['Speed'] >= speed_low) & (df['Speed'] <= speed_high)].copy()
    n_ep = df['pseudo_run_id'].nunique()
    print(f'  load_dataset @ {target_speed} m/s: {len(df)} baris, {n_ep} episodes')
    return df


def split_by_episode_stratified(df, num_classes=3, train_ratio=0.6, val_ratio=0.2, seed=42):
    rng = np.random.RandomState(seed)
    ep_to_label = df.groupby('pseudo_run_id')['label'].agg(lambda x: int(x.iloc[0])).to_dict()
    eps_by_class = {}
    for ep, lbl in ep_to_label.items():
        eps_by_class.setdefault(int(lbl), []).append(ep)
    train_eps, val_eps, test_eps = [], [], []
    for lbl in range(num_classes):
        eps = np.array(eps_by_class.get(lbl, []))
        if len(eps) == 0:
            continue
        rng.shuffle(eps)
        n = len(eps)
        n_train = int(round(train_ratio * n))
        n_val = int(round(val_ratio * n))
        if n >= 3:
            n_train = max(1, min(n - 2, n_train))
            n_val = max(1, min(n - n_train - 1, n_val))
        train_eps.extend(eps[:n_train].tolist())
        val_eps.extend(eps[n_train:n_train + n_val].tolist())
        test_eps.extend(eps[n_train + n_val:].tolist())
    return (df[df['pseudo_run_id'].isin(train_eps)].copy(),
            df[df['pseudo_run_id'].isin(val_eps)].copy(),
            df[df['pseudo_run_id'].isin(test_eps)].copy())


def compute_feature_stats(df):
    stat = {}
    for feat in FEATURES:
        stat[feat] = {
            'min': float(df[feat].min()),
            'max': float(df[feat].max())
        }
    return stat

def create_windows(df, window_sec):
    X_list, y_list = [], []
    for ep_id, grp in df.groupby('pseudo_run_id'):
        grp = grp.sort_values('Time').reset_index(drop=True)
        label = int(grp['label'].iloc[0])
        vals = grp[FEATURES].values
        T = len(vals)
        if T < window_sec:
            continue
        for start in range(0, T - window_sec + 1):
            X_list.append(vals[start:start + window_sec])
            y_list.append(label)
    return X_list, np.array(y_list, dtype=int)


def features_to_tokens(window_2d, feat_stats, n_bins=20):
    feature_names = ['SNR', 'RSSI', 'PDR', 'SPD', 'RSP']
    tokens = []
    for t in range(window_2d.shape[0]):
        for c, feat in enumerate(FEATURES):
            v = window_2d[t, c]
            fmin = feat_stats[feat]['min']
            fmax = feat_stats[feat]['max']
            if fmax - fmin < 1e-9:
                v_bin = 0
            else:
                v_bin = int(np.clip((v - fmin) / (fmax - fmin) * n_bins, 0, n_bins - 1))
            tokens.append(f'{feature_names[c]}{v_bin}')
    return ' '.join(tokens)


def compute_metrics(y_true, y_pred, num_classes=3):
    acc = float(accuracy_score(y_true, y_pred))
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    labels = list(range(num_classes))
    p_pc, r_pc, f1_pc, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average=None, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels).tolist()
    return dict(
        accuracy=acc, precision_macro=float(p), recall_macro=float(r), f1_macro=float(f1),
        precision_per_class=p_pc.tolist(), recall_per_class=r_pc.tolist(),
        f1_per_class=f1_pc.tolist(), confusion_matrix=cm
    )


print('Pipeline functions defined (incl. create_windows & range-filter load_dataset)')
print('  Fix 1: load_dataset pakai SPEED_RANGE (13-17 / 23-27)')
print('  Fix 3: create_windows sudah didefinisikan')


Pipeline functions defined (incl. create_windows & range-filter load_dataset)
  Fix 1: load_dataset pakai SPEED_RANGE (13-17 / 23-27)
  Fix 3: create_windows sudah didefinisikan


## Cell 4 — BERT Dataset Wrapper (PyTorch)

In [7]:
class BertJammingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels':         torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }


def run_bert_one(df_speed, seed, target_speed, results_dir):
    set_all_seeds(seed)
    out_path = os.path.join(results_dir, f'bert_wp_{target_speed}ms_seed{seed}.json')
    if os.path.exists(out_path):
        print(f'  [BERT-WP] seed {seed} @ {target_speed} m/s --- loaded from cache')
        with open(out_path) as f:
            return json.load(f)

    print(f'  [BERT-WP] seed {seed} @ {target_speed} m/s')
    train_df, val_df, test_df = split_by_episode_stratified(df_speed, num_classes=3, seed=seed)

    feat_stats = compute_feature_stats(train_df)
    print(f'    feat_stats dari train_df ({len(train_df)} rows) — no leakage')

    Xtr_l, ytr = create_windows(train_df, WINDOW_SEC)
    Xv_l,  yv  = create_windows(val_df,   WINDOW_SEC)
    Xte_l, yte = create_windows(test_df,  WINDOW_SEC)

    if min(len(Xtr_l), len(Xv_l), len(Xte_l)) == 0:
        print('    Empty windows, skip')
        return None

    tr_texts = [features_to_tokens(x, feat_stats, n_bins=20) for x in Xtr_l]
    v_texts  = [features_to_tokens(x, feat_stats, n_bins=20) for x in Xv_l]
    te_texts = [features_to_tokens(x, feat_stats, n_bins=20) for x in Xte_l]

    print(f'    Sample token (80 chars): {tr_texts[0][:80]}')
    print(f'    Train: {len(tr_texts)}, Val: {len(v_texts)}, Test: {len(te_texts)}')

    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    model     = DistilBertForSequenceClassification.from_pretrained(
                    'distilbert-base-uncased', num_labels=3
                ).to(device)

    train_ds = BertJammingDataset(tr_texts, ytr, tokenizer, MAX_LEN_TOK)
    val_ds   = BertJammingDataset(v_texts,  yv,  tokenizer, MAX_LEN_TOK)
    test_ds  = BertJammingDataset(te_texts, yte, tokenizer, MAX_LEN_TOK)

    train_loader = DataLoader(train_ds, batch_size=BERT_BATCH, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BERT_BATCH, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=BERT_BATCH, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=BERT_LR)

    best_val_loss    = float('inf')
    best_state       = None
    patience         = 3
    patience_counter = 0

    for epoch in range(BERT_EPOCHS):
        # Train
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            out = model(
                input_ids      = batch['input_ids'].to(device),
                attention_mask = batch['attention_mask'].to(device),
                labels         = batch['labels'].to(device)
            )
            out.loss.backward()
            optimizer.step()
            train_loss += out.loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                out = model(
                    input_ids      = batch['input_ids'].to(device),
                    attention_mask = batch['attention_mask'].to(device),
                    labels         = batch['labels'].to(device)
                )
                val_loss += out.loss.item()

        avg_tr  = train_loss / max(1, len(train_loader))
        avg_val = val_loss   / max(1, len(val_loader))
        print(f'    Epoch {epoch+1}/{BERT_EPOCHS} --- train: {avg_tr:.4f}, val: {avg_val:.4f}')

        if avg_val < best_val_loss:
            best_val_loss    = avg_val
            best_state       = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'    Early stop @ epoch {epoch+1}')
                break

    if best_state:
        model.load_state_dict(best_state)

    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in test_loader:
            out   = model(
                input_ids      = batch['input_ids'].to(device),
                attention_mask = batch['attention_mask'].to(device)
            )
            preds = torch.argmax(out.logits, dim=-1).cpu().numpy()
            all_preds.extend(preds.tolist())

    y_pred  = np.array(all_preds)
    metrics = compute_metrics(yte, y_pred, num_classes=3)
    print(f'    OK Acc: {metrics["accuracy"]:.4f} | F1: {metrics["f1_macro"]:.4f}')

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    result = dict(
        baseline='BERT_WordPiece', seed=seed, target_speed=target_speed,
        **metrics,
        y_true=yte.tolist(), y_pred=y_pred.tolist()
    )
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2)
    return result


print('BERT pipeline (Cell 4) defined --- OK')
print('  Fix 2: feat_stats dari train_df saja (no leakage)')
print('  Fix 5: patience=3')


BERT pipeline (Cell 4) defined --- OK
  Fix 2: feat_stats dari train_df saja (no leakage)
  Fix 5: patience=3


## Cell 4b — BERT Mitigation Attempt

In [8]:
from transformers import get_linear_schedule_with_warmup

BERT_MITIG_CONFIGS = {
    'warmup_lr2e5': {'lr': 2e-5, 'warmup_ratio': 0.1, 'epochs': 10, 'label': 'Warmup 10% + LR=2e-5'},
    'low_lr1e5':    {'lr': 1e-5, 'warmup_ratio': 0.0, 'epochs': 10, 'label': 'LR=1e-5 (no warmup)'},
}


def run_bert_mitig(df_speed, seed, target_speed, config_key, config, results_dir):
    set_all_seeds(seed)
    out_path = os.path.join(results_dir, f'bert_mitig_{config_key}_{target_speed}ms_seed{seed}.json')
    if os.path.exists(out_path):
        with open(out_path) as f:
            return json.load(f)

    print(f'  [BERT-MITIG {config_key}] seed {seed} @ {target_speed} m/s')
    train_df, val_df, test_df = split_by_episode_stratified(df_speed, num_classes=3, seed=seed)
    feat_stats = compute_feature_stats(train_df)

    Xtr_l, ytr = create_windows(train_df, WINDOW_SEC)
    Xv_l,  yv  = create_windows(val_df,   WINDOW_SEC)
    Xte_l, yte = create_windows(test_df,  WINDOW_SEC)
    if min(len(Xtr_l), len(Xv_l), len(Xte_l)) == 0:
        print('    Empty windows, skip'); return None

    tr_texts = [features_to_tokens(x, feat_stats, n_bins=20) for x in Xtr_l]
    v_texts  = [features_to_tokens(x, feat_stats, n_bins=20) for x in Xv_l]
    te_texts = [features_to_tokens(x, feat_stats, n_bins=20) for x in Xte_l]

    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    model     = DistilBertForSequenceClassification.from_pretrained(
                    'distilbert-base-uncased', num_labels=3).to(device)

    ds_tr = BertJammingDataset(tr_texts, ytr, tokenizer, MAX_LEN_TOK)
    ds_v  = BertJammingDataset(v_texts,  yv,  tokenizer, MAX_LEN_TOK)
    ds_te = BertJammingDataset(te_texts, yte, tokenizer, MAX_LEN_TOK)

    loader_tr = DataLoader(ds_tr, batch_size=BERT_BATCH, shuffle=True)
    loader_v  = DataLoader(ds_v,  batch_size=BERT_BATCH, shuffle=False)
    loader_te = DataLoader(ds_te, batch_size=BERT_BATCH, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=config['lr'])
    total_steps  = len(loader_tr) * config['epochs']
    warmup_steps = int(total_steps * config['warmup_ratio'])

    if warmup_steps > 0:
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
    else:
        scheduler = None

    best_val_loss = float('inf')
    best_state    = None
    patience_ctr  = 0
    patience      = 3

    for epoch in range(config['epochs']):
        model.train()
        tr_loss = 0.0
        for batch in loader_tr:
            optimizer.zero_grad()
            out = model(input_ids=batch['input_ids'].to(device),
                        attention_mask=batch['attention_mask'].to(device),
                        labels=batch['labels'].to(device))
            out.loss.backward(); optimizer.step()
            if scheduler: scheduler.step()
            tr_loss += out.loss.item()

        model.eval(); val_loss = 0.0
        with torch.no_grad():
            for batch in loader_v:
                out = model(input_ids=batch['input_ids'].to(device),
                            attention_mask=batch['attention_mask'].to(device),
                            labels=batch['labels'].to(device))
                val_loss += out.loss.item()

        avg_tr  = tr_loss  / max(1, len(loader_tr))
        avg_val = val_loss / max(1, len(loader_v))
        print(f'    Epoch {epoch+1}/{config["epochs"]} train={avg_tr:.4f} val={avg_val:.4f}')

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr  = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'    Early stop @ epoch {epoch+1}'); break

    if best_state: model.load_state_dict(best_state)
    model.eval(); all_preds = []
    with torch.no_grad():
        for batch in loader_te:
            out = model(input_ids=batch['input_ids'].to(device),
                        attention_mask=batch['attention_mask'].to(device))
            all_preds.extend(torch.argmax(out.logits, dim=-1).cpu().numpy().tolist())

    y_pred  = np.array(all_preds)
    metrics = compute_metrics(yte, y_pred, num_classes=3)
    print(f'    Acc: {metrics["accuracy"]:.4f} | F1: {metrics["f1_macro"]:.4f}')
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    result = dict(config_key=config_key, label=config['label'],
                  seed=seed, target_speed=target_speed,
                  **metrics, y_true=yte.tolist(), y_pred=y_pred.tolist())
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2)
    return result


print('BERT Mitigation functions defined')
print('  Config A: Warmup 10% + LR=2e-5')
print('  Config B: LR=1e-5 (no warmup)')


BERT Mitigation functions defined
  Config A: Warmup 10% + LR=2e-5
  Config B: LR=1e-5 (no warmup)


## Cell 4c-pre — Load Dataset untuk Mitigation

In [9]:
print("Loading datasets...")
df_25 = load_dataset(DATASET_25MS_PATH, target_speed=25)
df_15 = load_dataset(DATASET_15MS_PATH, target_speed=15)
print(f"df_25: {len(df_25)} rows | df_15: {len(df_15)} rows")
print("Siap untuk mitigation run.")

Loading datasets...
  load_dataset @ 25 m/s: 1195 baris, 154 episodes
  load_dataset @ 15 m/s: 1172 baris, 169 episodes
df_25: 1195 rows | df_15: 1172 rows
Siap untuk mitigation run.


## Cell 4c — Jalankan Semua Konfigurasi Mitigasi

In [10]:
mitig_results = {}

for config_key, config in BERT_MITIG_CONFIGS.items():
    print(f'\n=== Config: {config["label"]} ===')
    mitig_results[config_key] = {}
    for speed, df_sp, results_dir in [
        (25, df_25, RESULTS_25MS),
        (15, df_15, RESULTS_15MS),
    ]:
        runs = []
        for seed in SEEDS:
            res = run_bert_mitig(df_sp, seed=seed, target_speed=speed,
                                 config_key=config_key, config=config,
                                 results_dir=results_dir)
            if res: runs.append(res)
        mitig_results[config_key][speed] = runs

print()
print('=' * 90)
print('  BERT MITIGATION SUMMARY — Perbandingan Stabilitas')
print('=' * 90)
print(f'  {"Config":<30s} {"25m/s Acc":>18s} {"25m/s Std":>12s} {"15m/s Acc":>18s} {"15m/s Std":>12s}')
print('  ' + '-' * 90)
print(f'  {"Default (LR=2e-5, no warmup)":<30s} '
      f'  (lihat hasil Cell sebelumnya)')

for config_key, config in BERT_MITIG_CONFIGS.items():
    for speed in [25, 15]:
        runs = mitig_results[config_key].get(speed, [])
        if runs:
            accs = np.array([r['accuracy'] for r in runs]) * 100
            label = config['label'] if speed == 25 else ''
            if speed == 25:
                acc25 = f'{accs.mean():.2f}'; std25 = f'{accs.std(ddof=1):.2f}'
            else:
                acc15 = f'{accs.mean():.2f}'; std15 = f'{accs.std(ddof=1):.2f}'
    print(f'  {config["label"]:<30s} {acc25:>18s} {std25:>12s} {acc15:>18s} {std15:>12s}')

print()
print('  Interpretasi:')
print('  - Jika semua config std masih > 8%: instabilitas BERT inherent, klaim valid.')
print('  - Jika ada config std < 5%: update Section IV-E.3 dengan config stabil tersebut.')

# Simpan
mitig_summary = {}
for config_key, config in BERT_MITIG_CONFIGS.items():
    mitig_summary[config_key] = {'label': config['label']}
    for speed in [25, 15]:
        runs = mitig_results[config_key].get(speed, [])
        if runs:
            accs = np.array([r['accuracy'] for r in runs]) * 100
            mitig_summary[config_key][f'acc_{speed}ms'] = {
                'mean': float(accs.mean()), 'std': float(accs.std(ddof=1))
            }

import os
with open('/content/drive/MyDrive/Detector_Results/bert_mitigation_summary.json', 'w') as f:
    json.dump(mitig_summary, f, indent=2)
print('\nMitigation summary disimpan ke bert_mitigation_summary.json')



=== Config: Warmup 10% + LR=2e-5 ===

=== Config: LR=1e-5 (no warmup) ===

  BERT MITIGATION SUMMARY — Perbandingan Stabilitas
  Config                                  25m/s Acc    25m/s Std          15m/s Acc    15m/s Std
  ------------------------------------------------------------------------------------------
  Default (LR=2e-5, no warmup)     (lihat hasil Cell sebelumnya)
  Warmup 10% + LR=2e-5                        98.09         2.74              71.53        10.87
  LR=1e-5 (no warmup)                         98.09         2.74              70.09         9.81

  Interpretasi:
  - Jika semua config std masih > 8%: instabilitas BERT inherent, klaim valid.
  - Jika ada config std < 5%: update Section IV-E.3 dengan config stabil tersebut.

Mitigation summary disimpan ke bert_mitigation_summary.json


## Cell 5 — Run BERT @ 25 m/s (5 seeds)


In [11]:
import glob

old_files = glob.glob(os.path.join(RESULTS_25MS, 'bert_wp_25ms_seed*.json'))
for f in old_files:
    os.remove(f)
    print(f'Removed: {f}')

print('=' * 70)
print('  BERT WordPiece @ 25 m/s (Fixed binning + range filter)')
print('=' * 70)

df_25 = load_dataset(DATASET_25MS_PATH, target_speed=25)

bert_results_25 = []
for seed in SEEDS:
    res = run_bert_one(df_25, seed=seed, target_speed=25,
                       results_dir=RESULTS_25MS)
    if res:
        bert_results_25.append(res)

if bert_results_25:
    accs = np.array([r['accuracy'] for r in bert_results_25])
    f1s  = np.array([r['f1_macro'] for r in bert_results_25])
    print(f'\n@ 25 m/s FINAL:')
    print(f'  Accuracy: {accs.mean()*100:.2f} +/- {accs.std(ddof=1)*100:.2f}')
    print(f'  F1 macro: {f1s.mean()*100:.2f} +/- {f1s.std(ddof=1)*100:.2f}')


Removed: /content/drive/MyDrive/Detector_Results/baselines_25ms/bert_wp_25ms_seed42.json
Removed: /content/drive/MyDrive/Detector_Results/baselines_25ms/bert_wp_25ms_seed123.json
Removed: /content/drive/MyDrive/Detector_Results/baselines_25ms/bert_wp_25ms_seed456.json
Removed: /content/drive/MyDrive/Detector_Results/baselines_25ms/bert_wp_25ms_seed789.json
Removed: /content/drive/MyDrive/Detector_Results/baselines_25ms/bert_wp_25ms_seed2024.json
  BERT WordPiece @ 25 m/s (Fixed binning + range filter)
  load_dataset @ 25 m/s: 1195 baris, 154 episodes
  [BERT-WP] seed 42 @ 25 m/s
    feat_stats dari train_df (706 rows) — no leakage
    Sample token (80 chars): SNR19 RSSI19 PDR19 SPD19 RSP19 SNR17 RSSI13 PDR19 SPD19 RSP19 SNR15 RSSI4 PDR6 S
    Train: 303, Val: 74, Test: 135


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

    Epoch 1/10 --- train: 0.9451, val: 0.6061
    Epoch 2/10 --- train: 0.4610, val: 0.2018
    Epoch 3/10 --- train: 0.1863, val: 0.0784
    Epoch 4/10 --- train: 0.0989, val: 0.0427
    Epoch 5/10 --- train: 0.0700, val: 0.0275
    Epoch 6/10 --- train: 0.0532, val: 0.0216
    Epoch 7/10 --- train: 0.0393, val: 0.0150
    Epoch 8/10 --- train: 0.0397, val: 0.0172
    Epoch 9/10 --- train: 0.0294, val: 0.0098
    Epoch 10/10 --- train: 0.0382, val: 0.0094
    OK Acc: 1.0000 | F1: 1.0000
  [BERT-WP] seed 789 @ 25 m/s
    feat_stats dari train_df (808 rows) — no leakage
    Sample token (80 chars): SNR19 RSSI19 PDR19 SPD19 RSP19 SNR17 RSSI12 PDR19 SPD19 RSP19 SNR14 RSSI4 PDR6 S
    Train: 356, Val: 101, Test: 55


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    Epoch 1/10 --- train: 0.8878, val: 0.5132
    Epoch 2/10 --- train: 0.2512, val: 0.0843
    Epoch 3/10 --- train: 0.0746, val: 0.0370
    Epoch 4/10 --- train: 0.0502, val: 0.0261
    Epoch 5/10 --- train: 0.0253, val: 0.0143
    Epoch 6/10 --- train: 0.0315, val: 0.0103
    Epoch 7/10 --- train: 0.0190, val: 0.0084
    Epoch 8/10 --- train: 0.0102, val: 0.0060
    Epoch 9/10 --- train: 0.0082, val: 0.0049
    Epoch 10/10 --- train: 0.0059, val: 0.0040
    OK Acc: 0.9455 | F1: 0.9404
  [BERT-WP] seed 2024 @ 25 m/s
    feat_stats dari train_df (742 rows) — no leakage
    Sample token (80 chars): SNR19 RSSI19 PDR19 SPD18 RSP19 SNR17 RSSI12 PDR19 SPD18 RSP19 SNR14 RSSI4 PDR6 S
    Train: 324, Val: 102, Test: 86


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    Epoch 1/10 --- train: 0.8548, val: 0.3833
    Epoch 2/10 --- train: 0.2353, val: 0.0917
    Epoch 3/10 --- train: 0.0870, val: 0.0818
    Epoch 4/10 --- train: 0.0456, val: 0.0593
    Epoch 5/10 --- train: 0.0256, val: 0.0869
    Epoch 6/10 --- train: 0.0164, val: 0.0966
    Epoch 7/10 --- train: 0.0130, val: 0.0963
    Early stop @ epoch 7
    OK Acc: 1.0000 | F1: 1.0000

@ 25 m/s FINAL:
  Accuracy: 97.72 +/- 3.12
  F1 macro: 97.65 +/- 3.22


## Cell 6 — Run BERT @ 15 m/s (5 seeds)


In [12]:
old_files = glob.glob(os.path.join(RESULTS_15MS, 'bert_wp_15ms_seed*.json'))
for f in old_files:
    os.remove(f)
    print(f'Removed: {f}')

print('=' * 70)
print('  BERT WordPiece @ 15 m/s (Fixed binning + range filter)')
print('=' * 70)

df_15 = load_dataset(DATASET_15MS_PATH, target_speed=15)

bert_results_15 = []
for seed in SEEDS:
    res = run_bert_one(df_15, seed=seed, target_speed=15,
                       results_dir=RESULTS_15MS)
    if res:
        bert_results_15.append(res)

if bert_results_15:
    accs = np.array([r['accuracy'] for r in bert_results_15])
    f1s  = np.array([r['f1_macro'] for r in bert_results_15])
    print(f'\n@ 15 m/s FINAL:')
    print(f'  Accuracy: {accs.mean()*100:.2f} +/- {accs.std(ddof=1)*100:.2f}')
    print(f'  F1 macro: {f1s.mean()*100:.2f} +/- {f1s.std(ddof=1)*100:.2f}')


Removed: /content/drive/MyDrive/Detector_Results/baselines_15ms/bert_wp_15ms_seed42.json
Removed: /content/drive/MyDrive/Detector_Results/baselines_15ms/bert_wp_15ms_seed123.json
Removed: /content/drive/MyDrive/Detector_Results/baselines_15ms/bert_wp_15ms_seed456.json
Removed: /content/drive/MyDrive/Detector_Results/baselines_15ms/bert_wp_15ms_seed789.json
Removed: /content/drive/MyDrive/Detector_Results/baselines_15ms/bert_wp_15ms_seed2024.json
  BERT WordPiece @ 15 m/s (Fixed binning + range filter)
  load_dataset @ 15 m/s: 1172 baris, 169 episodes
  [BERT-WP] seed 42 @ 15 m/s
    feat_stats dari train_df (737 rows) — no leakage
    Sample token (80 chars): SNR19 RSSI19 PDR19 SPD12 RSP19 SNR17 RSSI14 PDR19 SPD12 RSP19 SNR14 RSSI7 PDR6 S
    Train: 299, Val: 82, Test: 95


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    Epoch 1/10 --- train: 0.9712, val: 0.6147
    Epoch 2/10 --- train: 0.5946, val: 0.4007
    Epoch 3/10 --- train: 0.5153, val: 0.3723
    Epoch 4/10 --- train: 0.4811, val: 0.3630
    Epoch 5/10 --- train: 0.4524, val: 0.3643
    Epoch 6/10 --- train: 0.4220, val: 0.3140
    Epoch 7/10 --- train: 0.3842, val: 0.2342
    Epoch 8/10 --- train: 0.3222, val: 0.2476
    Epoch 9/10 --- train: 0.2942, val: 0.2423
    Epoch 10/10 --- train: 0.3026, val: 0.1955
    OK Acc: 0.7789 | F1: 0.5698
  [BERT-WP] seed 123 @ 15 m/s
    feat_stats dari train_df (680 rows) — no leakage
    Sample token (80 chars): SNR19 RSSI19 PDR19 SPD12 RSP19 SNR17 RSSI14 PDR19 SPD12 RSP19 SNR14 RSSI7 PDR6 S
    Train: 270, Val: 136, Test: 70


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    Epoch 1/10 --- train: 0.9305, val: 0.5568
    Epoch 2/10 --- train: 0.5778, val: 0.3395
    Epoch 3/10 --- train: 0.5075, val: 0.2897
    Epoch 4/10 --- train: 0.4689, val: 0.2706
    Epoch 5/10 --- train: 0.4635, val: 0.2552
    Epoch 6/10 --- train: 0.4360, val: 0.2524
    Epoch 7/10 --- train: 0.4025, val: 0.2575
    Epoch 8/10 --- train: 0.3884, val: 0.3005
    Epoch 9/10 --- train: 0.3873, val: 0.2859
    Early stop @ epoch 9
    OK Acc: 0.6714 | F1: 0.5720
  [BERT-WP] seed 456 @ 15 m/s
    feat_stats dari train_df (587 rows) — no leakage
    Sample token (80 chars): SNR19 RSSI19 PDR19 SPD12 RSP19 SNR17 RSSI14 PDR19 SPD12 RSP19 SNR15 RSSI8 PDR6 S
    Train: 216, Val: 58, Test: 202


...

    Epoch 1/10 --- train: 1.0298, val: 0.8898
    Epoch 2/10 --- train: 0.6832, val: 0.6443
    Epoch 3/10 --- train: 0.5232, val: 0.6017
    Epoch 4/10 --- train: 0.4839, val: 0.5678
    Epoch 5/10 --- train: 0.4616, val: 0.5740
    Epoch 6/10 --- train: 0.4464, val: 0.5397
    Epoch 7/10 --- train: 0.4485, val: 0.5032
    Epoch 8/10 --- train: 0.4301, val: 0.5154
    Epoch 9/10 --- train: 0.4174, val: 0.5128
    Epoch 10/10 --- train: 0.4052, val: 0.5341
    Early stop @ epoch 10
    OK Acc: 0.8069 | F1: 0.7172
  [BERT-WP] seed 789 @ 15 m/s
    feat_stats dari train_df (837 rows) — no leakage
    Sample token (80 chars): SNR19 RSSI19 PDR19 SPD12 RSP19 SNR17 RSSI14 PDR19 SPD12 RSP19 SNR14 RSSI7 PDR6 S
    Train: 351, Val: 92, Test: 33


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    Epoch 1/10 --- train: 0.9108, val: 0.5203
    Epoch 2/10 --- train: 0.5043, val: 0.3230
    Epoch 3/10 --- train: 0.4077, val: 0.2688
    Epoch 4/10 --- train: 0.3696, val: 0.2543
    Epoch 5/10 --- train: 0.3650, val: 0.2580
    Epoch 6/10 --- train: 0.3532, val: 0.2618
    Epoch 7/10 --- train: 0.3419, val: 0.2642
    Early stop @ epoch 7
    OK Acc: 0.5455 | F1: 0.3529
  [BERT-WP] seed 2024 @ 15 m/s
    feat_stats dari train_df (724 rows) — no leakage
    Sample token (80 chars): SNR19 RSSI19 PDR19 SPD12 RSP19 SNR17 RSSI14 PDR19 SPD12 RSP19 SNR14 RSSI7 PDR6 S
    Train: 297, Val: 70, Test: 109


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    Epoch 1/10 --- train: 0.9537, val: 0.4988
    Epoch 2/10 --- train: 0.5835, val: 0.2677
    Epoch 3/10 --- train: 0.4753, val: 0.2477
    Epoch 4/10 --- train: 0.4519, val: 0.2005
    Epoch 5/10 --- train: 0.4224, val: 0.2227
    Epoch 6/10 --- train: 0.4011, val: 0.1611
    Epoch 7/10 --- train: 0.3935, val: 0.2536
    Epoch 8/10 --- train: 0.3671, val: 0.2331
    Epoch 9/10 --- train: 0.3526, val: 0.1612
    Early stop @ epoch 9
    OK Acc: 0.7156 | F1: 0.6988

@ 15 m/s FINAL:
  Accuracy: 70.37 +/- 10.31
  F1 macro: 58.22 +/- 14.55


## Cell 7 — Final Summary

In [13]:
from scipy import stats

def load_results(directory, baseline_prefix, target_speed):
    files = []
    for seed in SEEDS:
        fp = os.path.join(directory, f'{baseline_prefix}_{target_speed}ms_seed{seed}.json')
        if os.path.exists(fp):
            with open(fp) as f:
                files.append(json.load(f))
    return files


Detector_25 = '/content/drive/MyDrive/Detector_Results/seed_runs_25ms'
Detector_15 = '/content/drive/MyDrive/Detector_Results/seed_runs_15ms'

def load_detector(directory):
    results = []
    for seed in SEEDS:
        fp = os.path.join(directory, f'seed_{seed}_result.json')
        if os.path.exists(fp):
            with open(fp) as f:
                results.append(json.load(f))
    return results

detector_25 = load_detector(Detector_25)
detector_15 = load_detector(Detector_15)

rf_25    = load_results(RESULTS_25MS, 'rf_stat', 25)
rf_15    = load_results(RESULTS_15MS, 'rf_stat', 15)
lstm_25  = load_results(RESULTS_25MS, 'lstm_relu_iol1', 25)
lstm_15  = load_results(RESULTS_15MS, 'lstm_relu_iol1', 15)
bert_25  = load_results(RESULTS_25MS, 'bert_wp', 25)
bert_15  = load_results(RESULTS_15MS, 'bert_wp', 15)

print('Loaded results:')
print(f'  Detector 25m: {len(detector_25)}, Detector 15m: {len(detector_15)}')
print(f'  RF+Stat 25m: {len(rf_25)}, RF+Stat 15m: {len(rf_15)}')
print(f'  LSTM 25m: {len(lstm_25)}, LSTM 15m: {len(lstm_15)}')
print(f'  BERT 25m: {len(bert_25)}, BERT 15m: {len(bert_15)}')


def agg_pct(results):
    if len(results) == 0:
        return 'N/A'
    accs = np.array([r['accuracy'] for r in results]) * 100
    if len(accs) < 2:
        return f'{accs.mean():.2f}'
    return f'{accs.mean():.2f} +/- {accs.std(ddof=1):.2f}'


def paired_test(detector_results, baseline_results):
    if len(detector_results) < 2 or len(baseline_results) < 2:
        return None, None
    detector_sorted = sorted(detector_results, key=lambda r: r['seed'])
    base_sorted  = sorted(baseline_results, key=lambda r: r['seed'])
    if len(detector_sorted) != len(base_sorted):
        return None, None
    detector_accs = np.array([r['accuracy'] for r in detector_sorted])
    base_accs  = np.array([r['accuracy'] for r in base_sorted])
    t_stat, p_val = stats.ttest_rel(detector_accs, base_accs)
    return float(t_stat), float(p_val)


print('\n' + '=' * 90)
print('  TABLE VII --- Final Comparison (with BERT)')
print('=' * 90)
print()

header = f"{'Method':<25s} {'Lit 25m':<10s} {'Re-run 25m':<20s} {'p-val 25m':<12s} {'Lit 15m':<10s} {'Re-run 15m':<20s} {'p-val 15m':<12s}"
print(header)
print('-' * 115)

rows = [
    ('RF + Stat [14]',         rf_25,    rf_15,    '94.61', '80.04'),
    ('LSTM-RELU-IO-L1 [39]',  lstm_25,  lstm_15,  '95.67', '85.17'),
    ('BERT WordPiece [25]',   bert_25,  bert_15,  '96.25', '90.25'),
    ('Detector (Ours)',          detector_25, detector_15, '99.34', '95.45'),
]

for method, r25, r15, lit25, lit15 in rows:
    rerun_25 = agg_pct(r25)
    rerun_15 = agg_pct(r15)
    if method == 'Detector (Ours)':
        p25_str = '---'; p15_str = '---'
    else:
        _, p25 = paired_test(detector_25, r25)
        _, p15 = paired_test(detector_15, r15)
        p25_str = f'{p25:.4e}' if p25 is not None else 'N/A'
        p15_str = f'{p15:.4e}' if p15 is not None else 'N/A'
    print(f'{method:<25s} {lit25:<10s} {rerun_25:<20s} {p25_str:<12s} {lit15:<10s} {rerun_15:<20s} {p15_str:<12s}')

print()
print('Catatan: Lit = nilai original paper; Re-run = rata-rata 5 seeds eksperimen ini.')
print('         Detector re-run expected ≈ 98.17 (25 m/s) / 91.28 (15 m/s) sesuai Fase 2.')
print('Sig. levels: *** p<0.001, ** p<0.01, * p<0.05')

summary = {
    'detector_25': [r['accuracy'] for r in detector_25],
    'detector_15': [r['accuracy'] for r in detector_15],
    'rf_25': [r['accuracy'] for r in rf_25],
    'rf_15': [r['accuracy'] for r in rf_15],
    'lstm_25': [r['accuracy'] for r in lstm_25],
    'lstm_15': [r['accuracy'] for r in lstm_15],
    'bert_25': [r['accuracy'] for r in bert_25],
    'bert_15': [r['accuracy'] for r in bert_15],
}
with open('/content/drive/MyDrive/Detector_Results/tabel_VII_final.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\nFinal summary saved to /content/drive/MyDrive/Detector_Results/tabel_VII_final.json')


Loaded results:
  Detector 25m: 5, Detector 15m: 5
  RF+Stat 25m: 5, RF+Stat 15m: 5
  LSTM 25m: 5, LSTM 15m: 5
  BERT 25m: 5, BERT 15m: 5

  TABLE VII --- Final Comparison (with BERT)

Method                    Lit 25m    Re-run 25m           p-val 25m    Lit 15m    Re-run 15m           p-val 15m   
-------------------------------------------------------------------------------------------------------------------
RF + Stat [14]            94.61      100.00 +/- 0.00      3.8086e-02   80.04      72.28 +/- 9.01       7.9124e-03  
LSTM-RELU-IO-L1 [39]      95.67      67.78 +/- 27.01      6.0891e-02   85.17      51.12 +/- 28.03      3.1427e-02  
BERT WordPiece [25]       96.25      97.72 +/- 3.12       7.1401e-01   90.25      70.37 +/- 10.31      1.3099e-02  
Detector (Ours)           99.34      98.17 +/- 1.34       ---          95.45      91.28 +/- 3.87       ---         

Catatan: Lit = nilai original paper; Re-run = rata-rata 5 seeds eksperimen ini.
         Detector re-run expected ≈ 98